In [0]:
from pyspark.sql import functions as F
from pyspark.sql.window import Window
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

from sklearn.model_selection import train_test_split
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline
from sklearn.ensemble import RandomForestRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score

print("✓ Libraries Loaded")

In [0]:
fact_trip = spark.table("fact_trip")
dim_date = spark.table("dim_date")
dim_zone = spark.table("dim_zone")

In [0]:
%sql
CREATE OR REPLACE VIEW ml_trip_features_view AS
SELECT
    f.trip_key,
    f.booking_id,
    f.status,
    f.payment_type,
    f.booking_source,
    f.priority_level,
    f.fare_amount,
    f.distance,
    f.dispatch_to_arrival_minutes,
    f.expected_vs_actual_pickup_minutes_difference,
    f.trip_duration_minutes,

    d.day,
    d.month,
    d.quarter,
    d.year,
    d.weekend_flag,

    p.zone_code AS pickup_zone_code,
    p.latitude AS pickup_latitude,
    p.longitude AS pickup_longitude,

    dest.zone_code AS destination_zone_code,
    dest.latitude AS destination_latitude,
    dest.longitude AS destination_longitude

FROM fact_trip f
LEFT JOIN dim_date d
    ON f.date_key = d.date_key
LEFT JOIN dim_zone p
    ON f.pickup_zone_key = p.zone_key
LEFT JOIN dim_zone dest
    ON f.destination_zone_key = dest.zone_key
WHERE f.trip_duration_minutes IS NOT NULL

In [0]:
ml_df = spark.table("ml_trip_features_view")
ml_df.printSchema()
ml_df.show(5, truncate=False)

In [0]:
df = ml_df.toPandas()
df.head()
df.isnull().sum()

In [0]:
df['fare_per_distance'] = df['fare_amount'] / (df['distance'] + 1e-6)
df['pickup_delay_signal'] = df['expected_vs_actual_pickup_minutes_difference']
df['journey_urgency'] = df['dispatch_to_arrival_minutes'] + df['expected_vs_actual_pickup_minutes_difference']

df['lat_delta'] = (df['pickup_latitude'] - df['destination_latitude']).abs()
df['lon_delta'] = (df['pickup_longitude'] - df['destination_longitude']).abs()
df['location_distance_proxy'] = np.sqrt(df['lat_delta']**2 + df['lon_delta']**2)

df['dispatch_to_arrival_minutes'] = df['dispatch_to_arrival_minutes'].fillna(0)
df = df.dropna(subset=['destination_latitude', 'destination_longitude'])

In [0]:
TARGET = 'trip_duration_minutes'

feature_cols = [
    'priority_level',
    'status',
    'payment_type',
    'booking_source',
    'fare_amount',
    'distance',
    'expected_vs_actual_pickup_minutes_difference',
    'dispatch_to_arrival_minutes',
    'month',
    'quarter',
    'year',
    'weekend_flag',
    'pickup_latitude',
    'pickup_longitude',
    'destination_latitude',
    'destination_longitude',
    'fare_per_distance',
    'pickup_delay_signal',
    'journey_urgency',
    'lat_delta',
    'lon_delta',
    'location_distance_proxy'
]

In [0]:
df = df[(df[TARGET] > 0) & (df[TARGET] < 600)]
df = df[(df['distance'] > 0) & (df['distance'] < 200)]
df = df[(df['fare_amount'] > 0) & (df['fare_amount'] < 5000)]

In [0]:
X = df[feature_cols]
y = df[TARGET]

X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42
)

print("Train shape:", X_train.shape)
print("Test shape:", X_test.shape)

In [0]:
train_df = X_train.copy()
train_df[TARGET] = y_train.values

test_df = X_test.copy()
test_df[TARGET] = y_test.values

train_df.to_csv("train_trip_duration_features.csv", index=False)
test_df.to_csv("test_trip_duration_features.csv", index=False)